API para obtener evolucion del interes en distintos tipos de alimentos 

INSTALACION DE LA DEPENDENCIA PARA MANEJAR LA API

In [7]:
import pandas as pd
from pytrends.request import TrendReq
pytrend = TrendReq(hl="es-AR", tz=180)


SOLUCION AL PROBLEMA CON LA RES HTTP 429

In [8]:
import random, time
from pytrends.exceptions import ResponseError, TooManyRequestsError

def polite(fn, *args, retries=4, base_wait=30, pause=5, **kwargs):
    # Call fn(*args, **kwargs); back off and retry on HTTP 429.
    for attempt in range(retries):
        try:
            result = fn(*args, **kwargs)
            time.sleep(pause)  # breathing room before the next request
            return result
        except TooManyRequestsError:
            wait = base_wait * (2 ** attempt) + random.uniform(0, 5)
            print(f"429 hit. Waiting {wait:.0f}s (attempt {attempt + 1}/{retries})")
            time.sleep(wait)
        except ResponseError as e:
            # 400 (bad request) or 404 (retired endpoint): retrying won't help
            print(f"Google returned an error retrying won't fix: {e}")
            return None
    return None

Conceptos de interes en los ultimos 12 meses

In [9]:
from pytrends.request import TrendReq
import matplotlib.pyplot as plt 

pytrends = TrendReq(hl="es-AR", tz=180)


conceptos = ['alimentación saludable', 'productos orgánicos', 'veggie', 'productos sin TACC', 'café de especialidad']

pytrends.build_payload(
    kw_list=conceptos,
    timeframe="today 12-m",   # últimos 12 meses
    geo="AR",                # Argentina
)

df = pytrends.interest_over_time()
print(df.head())
df = df.drop(columns="isPartial")

df.plot(figsize=(12, 5), title="Evolución del interés en Argentina durante 2026", xlabel='Fecha', ylabel='Interes')


TooManyRequestsError: The request failed: Google returned a response with code 429

In [ ]:
pytrends_anios = TrendReq(hl="es-AR", tz=180)
pytrends_anios.build_payload(
    kw_list=conceptos,
    timeframe="today 5-y",   # últimos 5 años
    geo="AR",                # Argentina
)

df_anios = pytrends_anios.interest_over_time()
df_anios.to_csv('../data/raw/conceptos_interes.csv') #guardo el data frame en un csv para no depender de la api ya que esta anda mal
promedios = df_anios[conceptos].mean()
plt.figure(figsize=(8, 8))
plt.pie(
    promedios,
    labels=promedios.index,
    autopct="%1.1f%%",     # muestra el porcentaje en cada porción
    startangle=90,
)
plt.title("Interés relativo por concepto en Argentina (últimos 5 años)")
plt.show()


TooManyRequestsError: The request failed: Google returned a response with code 429